In [1]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required resources once.
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
stop_words.update(["rt", "amp"])

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def remove_urls(text):
  url_pattern = re.compile(r"https?://\S+|www\.\S+")
  return url_pattern.sub(" ", text)

def remove_emoji(text):
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(" ", text)

def removeunwanted_characters(document):
  # Remove mentions and hashtags text marker.
  document = re.sub(r"@[A-Za-z0-9_]+", " ", document)
  document = re.sub(r"#[A-Za-z0-9_]+", " ", document)
  # Keep letters, numbers and spaces only.
  document = re.sub(r"[^0-9A-Za-z ]", " ", document)
  # Normalize extra spaces.
  document = re.sub(r"\s+", " ", document).strip()
  return document

def remove_stopwords(tokens):
  return [token for token in tokens if token not in stop_words]

def lemmatization(tokens):
  return [lemmatizer.lemmatize(token, pos="v") for token in tokens]

def stemming(tokens):
  return [stemmer.stem(token) for token in tokens]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [2]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  Clean one text document and return normalized text.
  rule: "lemmatize" or "stem"
  """
  # Convert input to string and lowercase.
  data = str(dataset).lower()
  # Remove URLs.
  data = remove_urls(data)
  # Remove emojis.
  data = remove_emoji(data)
  # Remove all other unwanted characters.
  data = removeunwanted_characters(data)
  # Create tokens.
  tokens = data.split()
  # Remove stopwords.
  tokens = remove_stopwords(tokens)

  if rule == "lemmatize":
    tokens = lemmatization(tokens)
  elif rule == "stem":
    tokens = stemming(tokens)
  else:
    raise ValueError("Pick between lemmatize or stem")

  return " ".join(tokens)

# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [3]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Handle both possible dataset names in this workspace.
file_candidates = [
  Path("trum_tweet_sentiment_analysis.csv"),
  Path("trump_tweet_sentiment_analysis.csv")
]

data_path = next((p for p in file_candidates if p.exists()), None)
if data_path is None:
  raise FileNotFoundError("Could not find trump tweet sentiment dataset file.")

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Columns:", list(df.columns))

# Resolve text and label columns robustly.
text_col = "text" if "text" in df.columns else "content"
if text_col not in df.columns:
  raise KeyError("Expected a text/content column in dataset.")

if "label" in df.columns:
  label_col = "label"
elif "Sentiment" in df.columns:
  label_col = "Sentiment"
elif "sentiment" in df.columns:
  label_col = "sentiment"
else:
  raise KeyError("Expected one of: label, Sentiment, sentiment.")

df = df[[text_col, label_col]].dropna().copy()
df["clean_text"] = df[text_col].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

X_train, X_test, y_train, y_test = train_test_split(
  df["clean_text"],
  df[label_col],
  test_size=0.2,
  random_state=42,
  stratify=df[label_col]
  )

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

Loaded: trum_tweet_sentiment_analysis.csv
Columns: ['text', 'Sentiment']
              precision    recall  f1-score   support

           0       0.93      0.95      0.94    248842
           1       0.89      0.85      0.87    121183

    accuracy                           0.92    370025
   macro avg       0.91      0.90      0.91    370025
weighted avg       0.92      0.92      0.92    370025



In [4]:
from sklearn.metrics import accuracy_score, confusion_matrix

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

labels = sorted(pd.Series(y_test).unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(
  cm,
  index=[f"Actual_{label}" for label in labels],
  columns=[f"Predicted_{label}" for label in labels]
  )

print("Confusion Matrix:")
display(cm_df)

results_preview = pd.DataFrame({
  "original_text": df.loc[X_test.index, text_col].values,
  "clean_text": X_test.values,
  "actual": y_test.values,
  "predicted": y_pred
  })

print("Sample Predictions (first 10 test rows):")
display(results_preview.head(10))

Accuracy: 0.9177
Confusion Matrix:


,Predicted_0,Predicted_1
Actual_0,236465,12377
Actual_1,18072,103111


Sample Predictions (first 10 test rows):


,original_text,clean_text,actual,predicted
0,RT @samueloakford: Mar-a-Lago member who pays ...,mar lago member pay trump hundreds thousands d...,0,0
1,"RT @law_newz: Seriously, Arkansas? Even Trump ...",seriously arkansas even trump know sex marriag...,1,1
2,RT @Nigel_Farage: Bercow prefers North Korea t...,bercow prefer north korea president trump tota...,0,0
3,RT @realDrOlmo: BREAKING: Trump Was Right Loo...,break trump right look find raid mosque via,1,0
4,RT @McJesse: I EDITED ROBOCOP INTO TRUMP'S SPE...,edit robocop trump speech actually make sense,0,0
5,RT @BobbySacamano: I think this translates to ...,think translate f ck trump,0,0
6,RT @HirokoTabuchi: Can't help but notice that ...,help notice almost journalists color even less...,0,0
7,????Love it!! RT @_youhadonejob1: Dominican ne...,love dominican newspaper use picture alex bald...,1,1
8,RT @MMFlint: The Washington Post &amp; NY Time...,washington post ny time report trump know mont...,0,0
9,@iamlocaljo LIEEES. Trump supporters don't apo...,lieees trump supporters apologize,1,1


In [5]:
misclassified = results_preview[results_preview["actual"] != results_preview["predicted"]].copy()

print(f"Misclassified count: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified) / len(results_preview):.4f}")

display(misclassified[["original_text", "clean_text", "actual", "predicted"]].head(15))

Misclassified count: 30449
Misclassification rate: 0.0823


,original_text,clean_text,actual,predicted
3,RT @realDrOlmo: BREAKING: Trump Was Right Loo...,break trump right look find raid mosque via,1,0
10,Here's where Trump's travel ban stands right n...,trump travel ban stand right buzzfeed,1,0
15,RT @richard_littler: Since Trump blundered int...,since trump blunder power seem america develop...,0,1
16,Trump's Russia scandal means Sessions and his ...,trump russia scandal mean sessions justice dep...,0,1
19,Al Franken repeats senators' concern that Trum...,al franken repeat senators concern trump right...,1,0
21,RT @ClinicEscort: thank god the CDA's indecenc...,thank god cda indecency provision overturn bc ...,0,1
39,@TheLeadCNN @jaketapper how is GOP taking on T...,gop take trump talk work,0,1
48,"@ezlusztig Hi Elliott, my dad has fallen in lo...",hi elliott dad fall love trump hook line sinke...,0,1
49,RT @HuffingtonPost: Solar energy created 1 in ...,solar energy create 1 every 50 new u job last ...,0,1
58,RT @Zanting: President Donald Trump (@POTUS): ...,president donald trump undisputed leader take ...,1,0


In [6]:
from sklearn.metrics import classification_report

report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).T
display(report_df.round(4))

,precision,recall,f1-score,support
0,0.9290,0.9503,0.9395,248842.0000
1,0.8928,0.8509,0.8713,121183.0000
accuracy,0.9177,0.9177,0.9177,0.9177
macro avg,0.9109,0.9006,0.9054,370025.0000
weighted avg,0.9172,0.9177,0.9172,370025.0000


In [8]:
def predict_sentiment(raw_text):
  cleaned = text_cleaning_pipeline(raw_text, rule="lemmatize")
  vec = tfidf.transform([cleaned])
  pred = model.predict(vec)[0]
  return int(pred), cleaned

sample_texts = [
  "Trump is doing an excellent job for the economy.",
  "I strongly disagree with Trump policies and speeches."
]

for t in sample_texts:
  pred, cleaned = predict_sentiment(t)
  print("Text:", t)
  print("Cleaned:", cleaned)
  print("Predicted label:", pred)
  print("-" * 60)

Text: Trump is doing an excellent job for the economy.
Cleaned: trump excellent job economy
Predicted label: 1
------------------------------------------------------------
Text: I strongly disagree with Trump policies and speeches.
Cleaned: strongly disagree trump policies speeches
Predicted label: 0
------------------------------------------------------------
